In [1]:
pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 129.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 144.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [2]:
pip install folium

In [3]:
pip install numpy

In [4]:
pip install pandas


In [5]:
pip install streamlit-folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.9/536.9 kB 16.3 MB/s  0:00:00


In [6]:
!pip install -q streamlit folium streamlit-folium pandas numpy
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [7]:
!pip install -q requests

In [8]:
%%writefile app.py

import streamlit as st
import folium
from streamlit_folium import st_folium
import requests
import pandas as pd
from datetime import datetime, timedelta

st.set_page_config(
    page_title="Field Shift – Rotation Explorer",
    page_icon="🌱",
    layout="wide"
)

st.title("🌱 Field Shift: Interactive Rotation Scenario Explorer")
st.caption("NASA Space Apps Challenge 2026 | Using NASA POWER data")

# -------------------------------------------------
# Crop knowledge base
# -------------------------------------------------
CROP_TRAITS = {
    "Corn": {"water": 8, "soil": 3, "erosion": 3, "nitrogen": 1, "income": 8,
             "desc": "High water and nitrogen demand. Good market value but can degrade soil if repeated."},
    "Soybean": {"water": 5, "soil": 7, "erosion": 5, "nitrogen": 9, "income": 7,
                "desc": "Nitrogen-fixing legume. Improves soil fertility and reduces fertilizer need."},
    "Wheat": {"water": 4, "soil": 5, "erosion": 6, "nitrogen": 3, "income": 5,
              "desc": "Moderate water use. Good residue cover that protects soil."},
    "Alfalfa": {"water": 6, "soil": 9, "erosion": 9, "nitrogen": 9, "income": 6,
                "desc": "Deep-rooted perennial. Excellent soil builder, nitrogen fixer, and erosion controller."},
    "Cover Crop (Rye + Clover)": {"water": 3, "soil": 9, "erosion": 9, "nitrogen": 7, "income": 2,
                                  "desc": "Protects bare soil, builds organic matter, reduces erosion."},
    "Cotton": {"water": 7, "soil": 2, "erosion": 3, "nitrogen": 2, "income": 7,
               "desc": "High input crop. Can leave soil exposed and nutrient-depleted."},
    "Sorghum": {"water": 3, "soil": 5, "erosion": 6, "nitrogen": 3, "income": 5,
                "desc": "Drought-tolerant. Lower water demand than corn."}
}

def score_rotation(crops, priorities):
    scores = {"water": 0, "soil": 0, "erosion": 0, "nitrogen": 0, "income": 0}
    for crop in crops:
        traits = CROP_TRAITS.get(crop, {})
        for key in scores:
            scores[key] += traits.get(key, 5)
    n = max(len(crops), 1)
    for key in scores:
        scores[key] = round(scores[key] / n, 1)

    total = (
        scores["water"] * priorities["water"] +
        scores["soil"] * priorities["soil"] +
        scores["erosion"] * priorities["erosion"] +
        scores["nitrogen"] * priorities["nitrogen"] +
        scores["income"] * priorities["income"]
    ) / (sum(priorities.values()) + 1e-6)
    return scores, round(total, 1)

def get_explanation(name, crops):
    if name == "Current":
        return "This is your existing rotation. Used as the baseline for comparison."
    elif name == "Soil Health Focus":
        return f"This rotation ({' → '.join(crops)}) prioritizes soil building, nitrogen fixation, and erosion control."
    elif name == "Water Efficient Focus":
        return f"This sequence ({' → '.join(crops)}) reduces water demand and keeps living cover longer — useful under dry conditions."
    return ""

# -------------------------------------------------
# NASA POWER function
# -------------------------------------------------
@st.cache_data(ttl=3600)
def get_nasa_power_data(lat, lon):
    """Fetch recent temperature and rainfall from NASA POWER"""
    end_date = datetime.utcnow()
    start_date = end_date - timedelta(days=30)

    start_str = start_date.strftime("%Y%m%d")
    end_str = end_date.strftime("%Y%m%d")

    url = (
        f"https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?parameters=T2M,PRECTOTCORR"
        f"&community=AG"
        f"&longitude={lon}"
        f"&latitude={lat}"
        f"&start={start_str}"
        f"&end={end_str}"
        f"&format=JSON"
    )

    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        data = response.json()

        properties = data["properties"]["parameter"]
        temps = list(properties["T2M"].values())
        rains = list(properties["PRECTOTCORR"].values())

        # Filter out missing values (-999)
        temps = [t for t in temps if t != -999]
        rains = [r for r in rains if r != -999]

        avg_temp = round(sum(temps) / len(temps), 1) if temps else None
        total_rain = round(sum(rains), 1) if rains else None
        max_temp = round(max(temps), 1) if temps else None

        return {
            "avg_temp": avg_temp,
            "max_temp": max_temp,
            "total_rain": total_rain,
            "days": len(temps)
        }
    except Exception as e:
        return {"error": str(e)}

# -------------------------------------------------
# Sidebar
# -------------------------------------------------
with st.sidebar:
    st.header("📍 Field Location")
    lat = st.number_input("Latitude", min_value=-90.0, max_value=90.0, value=41.60, step=0.01, format="%.4f")
    lon = st.number_input("Longitude", min_value=-180.0, max_value=180.0, value=-93.60, step=0.01, format="%.4f")

    st.header("🔄 Current Rotation")
    crop_options = list(CROP_TRAITS.keys())
    year1 = st.selectbox("Year 1 crop", options=crop_options, index=0)
    year2 = st.selectbox("Year 2 crop", options=crop_options, index=1)
    year3 = st.selectbox("Year 3 (optional)", options=["None"] + crop_options, index=0)

    st.header("🎯 Your Priorities")
    p_water = st.slider("Conserve water", 0, 10, 7)
    p_soil = st.slider("Build soil health", 0, 10, 8)
    p_erosion = st.slider("Reduce erosion", 0, 10, 6)
    p_nitrogen = st.slider("Add nitrogen / cut fertilizer", 0, 10, 6)
    p_income = st.slider("Income / forage value", 0, 10, 5)

priorities = {
    "water": p_water,
    "soil": p_soil,
    "erosion": p_erosion,
    "nitrogen": p_nitrogen,
    "income": p_income
}

current = [year1, year2]
if year3 != "None":
    current.append(year3)

soil_focus = ["Soybean", "Alfalfa", "Cover Crop (Rye + Clover)"]
water_focus = ["Sorghum", "Wheat", "Cover Crop (Rye + Clover)"]

# -------------------------------------------------
# Map
# -------------------------------------------------
st.subheader("Field Location")
m = folium.Map(location=[lat, lon], zoom_start=11)
folium.Marker([lat, lon], tooltip="Selected Field", popup="Your field").add_to(m)
folium.Circle(location=[lat, lon], radius=500, color="#2e7d32", fill=True, fill_opacity=0.25).add_to(m)
st_folium(m, width=1000, height=380, key="main_field_map")

# -------------------------------------------------
# NASA POWER Climate Context
# -------------------------------------------------
st.markdown("---")
st.subheader("📡 NASA POWER Climate Context (Last 30 Days)")

with st.spinner("Fetching NASA POWER data..."):
    climate = get_nasa_power_data(lat, lon)

if "error" in climate:
    st.error(f"Could not fetch NASA data: {climate['error']}")
else:
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Avg Temperature", f"{climate['avg_temp']} °C")
    col2.metric("Max Temperature", f"{climate['max_temp']} °C")
    col3.metric("Total Rainfall", f"{climate['total_rain']} mm")
    col4.metric("Days of data", climate['days'])

    st.caption("Source: NASA POWER | Parameters: T2M (temperature) + PRECTOTCORR (precipitation)")

# -------------------------------------------------
# Rotation Comparison
# -------------------------------------------------
st.markdown("---")
st.subheader("🔄 Rotation Strategy Comparison")

strategies = [
    ("Current", current),
    ("Soil Health Focus", soil_focus),
    ("Water Efficient Focus", water_focus)
]

cols = st.columns(3)

for idx, (name, rotation) in enumerate(strategies):
    scores, total = score_rotation(rotation, priorities)
    explanation = get_explanation(name, rotation)

    with cols[idx]:
        st.markdown(f"### {name}")
        st.markdown(f"**Sequence:** {' → '.join(rotation)}")
        st.metric("Fit Score", f"{total}")

        st.write("**Breakdown**")
        st.write(f"• Water: `{scores['water']}/10`")
        st.write(f"• Soil health: `{scores['soil']}/10`")
        st.write(f"• Erosion: `{scores['erosion']}/10`")
        st.write(f"• Nitrogen: `{scores['nitrogen']}/10`")
        st.write(f"• Income: `{scores['income']}/10`")

        with st.expander("Why this rotation?"):
            st.write(explanation)

# Best match
totals = [(name, score_rotation(rot, priorities)[1]) for name, rot in strategies]
best_name, best_score = max(totals, key=lambda x: x[1])
st.success(f"**Best match for your priorities:** {best_name} (score {best_score})")

st.info("NASA POWER data is now connected. Next possible upgrades: better explanations linked to climate, or full SMAP/NDVI with Google Earth Engine.")

Writing app.py


In [9]:
!pip install -q streamlit folium streamlit-folium requests pyngrok

In [10]:
from pyngrok import ngrok
ngrok.set_auth_token("3M9loU1FGqu5lIio748dBd7mro8_3bpmSSwQnZqhrL3rEggCx")
print("Token set")

Token set


In [11]:
!pkill -f streamlit || true

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true > /content/st.log 2>&1 &

import time
time.sleep(12)
print("Streamlit started. Log:")
!cat /content/st.log

^C
Streamlit started. Log:


2026-11-20 12:20:35.045 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.187.236.193:8501



In [12]:
from pyngrok import ngrok

# Close any old tunnels first
ngrok.kill()

public_url = ngrok.connect(8501)
print("\n=================================")
print("Your app link:")
print(public_url)
print("=================================")


Your app link:
NgrokTunnel: "https://wispy-detached-sandfish.ngrok-free.dev" -> "http://localhost:8501"
